### Load Bronze, Silver and quarantine tables

In [0]:
from pyspark.sql import functions as F

customers_bronze = spark.table("workspace.commerce_dataset.bronze_customers")
products_bronze = spark.table("workspace.commerce_dataset.bronze_products")
orders_bronze = spark.table("workspace.commerce_dataset.bronze_orders")
order_items_bronze = spark.table("workspace.commerce_dataset.bronze_order_items")
payments_bronze = spark.table("workspace.commerce_dataset.bronze_payments")

customers_silver = spark.table("workspace.commerce_dataset.silver_customers")
products_silver = spark.table("workspace.commerce_dataset.silver_products")
orders_silver = spark.table("workspace.commerce_dataset.silver_orders")
order_items_silver = spark.table("workspace.commerce_dataset.silver_order_items")
payments_silver = spark.table("workspace.commerce_dataset.silver_payments")

customers_quarantine = spark.table("workspace.commerce_dataset.quarantine_customers")
products_quarantine = spark.table("workspace.commerce_dataset.quarantine_products")
orders_quarantine = spark.table("workspace.commerce_dataset.quarantine_orders")
order_items_quarantine = spark.table("workspace.commerce_dataset.quarantine_order_items")
payments_quarantine = spark.table("workspace.commerce_dataset.quarantine_payments")


### Calculate expected order value

In [0]:
expected_order_value = (
    order_items_silver
    .groupBy("order_id")
    .agg(
        F.round(F.sum("line_total"), 2).alias("expected_order_value"),
        F.count("*").alias("item_count")
    )
)

print("Expected order value from trusted order items:")
display(expected_order_value.limit(20))


### Summarize payment activity

In [0]:
payment_activity = (
    payments_silver
    .groupBy("order_id")
    .agg(
        F.round(F.sum(F.when(F.col("payment_status") == "paid", F.col("amount")).otherwise(F.lit(0))), 2).alias("paid_amount"),
        F.round(F.sum(F.when(F.col("payment_status") == "refunded", F.col("amount")).otherwise(F.lit(0))), 2).alias("refund_amount"),
        F.sum(F.when(F.col("payment_status") == "paid", 1).otherwise(0)).alias("paid_payment_count"),
        F.sum(F.when(F.col("payment_status") == "refunded", 1).otherwise(0)).alias("refund_payment_count"),
        F.sum(F.when(F.col("payment_status") == "failed", 1).otherwise(0)).alias("failed_payment_count")
    )
)

rejected_payment_activity = payments_quarantine.filter(F.col("order_id").isNotNull()).groupBy("order_id").agg(F.count("*").alias("rejected_payment_count"))

print("Trusted payment activity:")
display(payment_activity.limit(20))


### Build Finance reconciliation

In [0]:
finance_reconciliation = (
    orders_silver
    .select("order_id", "customer_id", "order_date", "shipping_city", "order_status")
    .join(expected_order_value, on="order_id", how="left")
    .join(payment_activity, on="order_id", how="left")
    .join(rejected_payment_activity, on="order_id", how="left")
    .withColumn("expected_order_value", F.coalesce(F.col("expected_order_value"), F.lit(0)))
    .withColumn("paid_amount", F.coalesce(F.col("paid_amount"), F.lit(0)))
    .withColumn("refund_amount", F.coalesce(F.col("refund_amount"), F.lit(0)))
    .withColumn("paid_payment_count", F.coalesce(F.col("paid_payment_count"), F.lit(0)))
    .withColumn("refund_payment_count", F.coalesce(F.col("refund_payment_count"), F.lit(0)))
    .withColumn("failed_payment_count", F.coalesce(F.col("failed_payment_count"), F.lit(0)))
    .withColumn("rejected_payment_count", F.coalesce(F.col("rejected_payment_count"), F.lit(0)))
    .withColumn("reconciliation_difference", F.round(F.col("paid_amount") - F.col("expected_order_value"), 2))
    .withColumn(
        "reconciliation_status",
        F.when((F.col("order_status") == "refunded") | (F.col("refund_payment_count") > 0), "REFUNDED")
        .when(F.col("expected_order_value") <= 0, "NON_STANDARD")
        .when((F.col("rejected_payment_count") > 0) & (F.col("paid_amount") <= 0), "NON_STANDARD")
        .when(F.col("paid_amount") <= 0, "MISSING_PAYMENT")
        .when((F.col("paid_amount") >= F.col("expected_order_value") - 0.01) & (F.col("paid_amount") <= F.col("expected_order_value") + 0.01), "MATCHED")
        .when(F.col("paid_amount") < F.col("expected_order_value") - 0.01, "UNDERPAID")
        .otherwise("OVERPAID")
    )
    .withColumn("trusted_revenue", F.when((F.col("order_status") == "completed") & (F.col("reconciliation_status") == "MATCHED"), F.col("expected_order_value")).otherwise(F.lit(0)))
)

print("FINANCE RECONCILIATION")
display(finance_reconciliation.limit(20))


### Summarize reconciliation results

In [0]:
reconciliation_summary = (
    finance_reconciliation
    .groupBy("reconciliation_status")
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("expected_order_value"), 2).alias("expected_order_value"),
        F.round(F.sum("paid_amount"), 2).alias("paid_amount"),
        F.round(F.sum("trusted_revenue"), 2).alias("trusted_revenue")
    )
    .orderBy(F.desc("order_count"))
)

print("RECONCILIATION STATUS COUNTS")
display(reconciliation_summary)

reconciliation_investigation = (
    finance_reconciliation
    .filter(F.col("reconciliation_status").isin("UNDERPAID", "OVERPAID"))
    .withColumn("difference_to_investigate", F.when(F.col("reconciliation_difference") < 0, -F.col("reconciliation_difference")).otherwise(F.col("reconciliation_difference")))
    .orderBy(F.desc("difference_to_investigate"))
)

print("LARGEST RECONCILIATION DIFFERENCES")
display(reconciliation_investigation.select("order_id", "customer_id", "order_status", "expected_order_value", "paid_amount", "reconciliation_difference", "reconciliation_status").limit(20))


### Calculate trusted revenue

In [0]:
trusted_revenue_orders = (
    finance_reconciliation
    .filter((F.col("order_status") == "completed") & (F.col("reconciliation_status") == "MATCHED"))
    .join(customers_silver.select("customer_id", "customer_name", "customer_type"), on="customer_id", how="inner")
)

trusted_revenue_stats = trusted_revenue_orders.agg(
    F.count("*").alias("trusted_revenue_orders"),
    F.round(F.sum("trusted_revenue"), 2).alias("trusted_revenue")
).first()

trusted_revenue_order_count = trusted_revenue_stats["trusted_revenue_orders"] or 0
trusted_revenue_total = trusted_revenue_stats["trusted_revenue"] or 0

print(f"Trusted revenue orders: {trusted_revenue_order_count:,}")
print(f"Trusted revenue: ${trusted_revenue_total:,.2f}")


### Analyze revenue by date and city

In [0]:
revenue_by_date_city = (
    trusted_revenue_orders
    .groupBy("order_date", "shipping_city")
    .agg(
        F.round(F.sum("trusted_revenue"), 2).alias("trusted_revenue"),
        F.count("*").alias("order_count")
    )
    .orderBy("order_date", F.desc("trusted_revenue"))
)

print("TRUSTED REVENUE BY DATE AND CITY")
display(revenue_by_date_city)


### Analyze product and category revenue

In [0]:
trusted_product_lines = (
    trusted_revenue_orders
    .select("order_id")
    .join(order_items_silver.select("order_id", "product_id", "quantity", "line_total"), on="order_id", how="inner")
    .join(products_silver.select("product_id", "product_name", "category"), on="product_id", how="inner")
)

revenue_by_product = (
    trusted_product_lines
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.round(F.sum("line_total"), 2).alias("trusted_revenue"),
        F.sum("quantity").alias("units_sold")
    )
    .orderBy(F.desc("trusted_revenue"))
)

revenue_by_category = (
    trusted_product_lines
    .groupBy("category")
    .agg(
        F.round(F.sum("line_total"), 2).alias("trusted_revenue"),
        F.sum("quantity").alias("units_sold")
    )
    .orderBy(F.desc("trusted_revenue"))
)

print("TOP PRODUCTS BY TRUSTED REVENUE")
display(revenue_by_product.limit(20))

print("TRUSTED REVENUE BY CATEGORY")
display(revenue_by_category)


### Analyze customer revenue

In [0]:
revenue_by_customer = (
    trusted_revenue_orders
    .groupBy("customer_id", "customer_name", "customer_type")
    .agg(
        F.round(F.sum("trusted_revenue"), 2).alias("trusted_revenue"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("trusted_revenue"))
)

revenue_by_customer_type = (
    trusted_revenue_orders
    .groupBy("customer_type")
    .agg(
        F.round(F.sum("trusted_revenue"), 2).alias("trusted_revenue"),
        F.count("*").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(F.desc("trusted_revenue"))
)

print("TOP CUSTOMERS BY TRUSTED REVENUE")
display(revenue_by_customer.limit(20))

print("TRUSTED REVENUE BY CUSTOMER TYPE")
display(revenue_by_customer_type)


### Calculate source-to-Silver quality

In [0]:
customer_raw_count = customers_bronze.count()
product_raw_count = products_bronze.count()
order_raw_count = orders_bronze.count()
order_item_raw_count = order_items_bronze.count()
payment_raw_count = payments_bronze.count()

customer_trusted_count = customers_silver.count()
product_trusted_count = products_silver.count()
order_trusted_count = orders_silver.count()
order_item_trusted_count = order_items_silver.count()
payment_trusted_count = payments_silver.count()

customer_rejected_count = customers_quarantine.count()
product_rejected_count = products_quarantine.count()
order_rejected_count = orders_quarantine.count()
order_item_rejected_count = order_items_quarantine.count()
payment_rejected_count = payments_quarantine.count()

quality_rows = [
    ("customers", customer_raw_count, customer_rejected_count, customer_trusted_count, round(customer_trusted_count / customer_raw_count * 100, 2)),
    ("products", product_raw_count, product_rejected_count, product_trusted_count, round(product_trusted_count / product_raw_count * 100, 2)),
    ("orders", order_raw_count, order_rejected_count, order_trusted_count, round(order_trusted_count / order_raw_count * 100, 2)),
    ("order_items", order_item_raw_count, order_item_rejected_count, order_item_trusted_count, round(order_item_trusted_count / order_item_raw_count * 100, 2)),
    ("payments", payment_raw_count, payment_rejected_count, payment_trusted_count, round(payment_trusted_count / payment_raw_count * 100, 2))
]

quality_summary = spark.createDataFrame(quality_rows, ["entity_name", "raw_count", "rejected_count", "trusted_count", "silver_success_rate_pct"])

total_raw_count = customer_raw_count + product_raw_count + order_raw_count + order_item_raw_count + payment_raw_count
total_trusted_count = customer_trusted_count + product_trusted_count + order_trusted_count + order_item_trusted_count + payment_trusted_count
overall_success_rate = total_trusted_count / total_raw_count * 100

print("DATA QUALITY SUMMARY")
display(quality_summary)
print(f"Overall source-to-Silver success rate: {overall_success_rate:.2f}%")


### Save the Gold Delta tables

In [0]:
finance_reconciliation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_finance_reconciliation")
trusted_revenue_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_trusted_revenue_orders")
revenue_by_date_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_revenue_by_date_city")
revenue_by_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_revenue_by_product")
revenue_by_category.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_revenue_by_category")
revenue_by_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_revenue_by_customer")
revenue_by_customer_type.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_revenue_by_customer_type")
reconciliation_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_reconciliation_summary")
reconciliation_investigation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_reconciliation_investigation")
quality_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.gold_data_quality_summary")

print("Gold Delta tables saved successfully.")


### Inspect the Spark execution plan

In [0]:
trusted_product_lines.explain()

print("Spark builds the transformations first and executes them when an action such as display(), count() or a table write is called.")


### Trace one order through the pipeline

In [0]:
TRACE_ORDER_ID = 600002

print(f"BRONZE - ORDER {TRACE_ORDER_ID}")
display(orders_bronze.filter(F.col("order_id") == TRACE_ORDER_ID))
display(order_items_bronze.filter(F.col("order_id") == TRACE_ORDER_ID))
display(payments_bronze.filter(F.col("order_id") == TRACE_ORDER_ID))

print(f"SILVER - ORDER {TRACE_ORDER_ID}")
display(orders_silver.filter(F.col("order_id") == TRACE_ORDER_ID))
display(order_items_silver.filter(F.col("order_id") == TRACE_ORDER_ID))
display(payments_silver.filter(F.col("order_id") == TRACE_ORDER_ID))

print(f"QUARANTINE - ORDER {TRACE_ORDER_ID}")
display(orders_quarantine.filter(F.col("order_id") == TRACE_ORDER_ID))
display(order_items_quarantine.filter(F.col("order_id") == TRACE_ORDER_ID))
display(payments_quarantine.filter(F.col("order_id") == TRACE_ORDER_ID))

print(f"GOLD - ORDER {TRACE_ORDER_ID}")
display(finance_reconciliation.filter(F.col("order_id") == TRACE_ORDER_ID))


### Show the final management summary

In [0]:
print("FINAL MANAGEMENT VIEW")
print(f"Trusted revenue orders: {trusted_revenue_order_count:,}")
print(f"Trusted revenue: ${trusted_revenue_total:,.2f}")
print(f"Overall source-to-Silver success rate: {overall_success_rate:.2f}%")

print("Reconciliation counts:")
display(finance_reconciliation.groupBy("reconciliation_status").count().orderBy(F.desc("count")))

print("Finance should use the trusted Gold revenue outputs instead of calculating revenue directly from the raw CSV files.")
